# 04 — Market Making Engine
Cartea & Wang (2020) alpha-integrated market making for binary options.

In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from config import *
from utils import (
    reservation_price, compute_spread, compute_quotes,
    check_toxicity, time_spread_multiplier
)

DATA_DIR = Path('../data')

In [ ]:
features = pd.read_parquet(DATA_DIR / 'features.parquet')
regimes = pd.read_parquet(DATA_DIR / 'regimes.parquet')
print(f'Loaded {len(features):,} bars')

## Alpha Signal & Mode Switching

In [ ]:
def compute_mode(alpha):
    """Determine trading mode from alpha signal."""
    abs_alpha = abs(alpha)
    if abs_alpha < ALPHA_THRESHOLD_1:
        return 'two_sided'       # provide liquidity symmetrically
    elif abs_alpha < ALPHA_THRESHOLD_2:
        return 'one_sided'       # post only in alpha direction
    else:
        return 'directional'     # market orders in alpha direction


def compute_action(p_up, vpin, vpin_mean, vpin_std, regime_score,
                   time_remaining, inventory, capital):
    """Full quoting decision combining all signals.

    Returns dict with bid, ask, mode, and action details.
    """
    alpha = p_up - 0.5
    mode = compute_mode(alpha)

    # Toxicity check
    tox = check_toxicity(vpin, vpin_mean, vpin_std)
    if tox == 'WITHDRAW':
        return {'mode': 'withdraw', 'bid': None, 'ask': None, 'reason': 'VPIN toxic'}

    # Risk aversion (elevated for unhedgeable binary)
    q_max = capital / max(p_up, 1 - p_up)
    gamma = 1.0 / (q_max * max(p_up, 1 - p_up))

    # Reservation price
    res_p = reservation_price(p_up, inventory, gamma, time_remaining)

    # Spread
    k_param = 1.5  # order arrival intensity
    base_spread = compute_spread(p_up, gamma, k_param, time_remaining)

    # Time-dependent multiplier
    t_mult = time_spread_multiplier(time_remaining, p_up)
    if t_mult == np.inf:
        return {'mode': 'cease', 'bid': None, 'ask': None, 'reason': 'ATM near expiry'}

    # Toxicity spread adjustment
    tox_mult = {'NORMAL': 1.0, 'WIDEN': 2.0, 'WIDEN_MAX': 4.0}.get(tox, 1.0)

    # Regime spread adjustment
    regime_mult = 1.0
    if regime_score > 0.3:
        regime_mult = 1.5 + regime_score  # wider in trending

    half_spread = base_spread * t_mult * tox_mult * regime_mult / 2

    # Compute quotes
    bid, ask = compute_quotes(res_p, half_spread, alpha)

    # Inventory limits
    if inventory >= q_max:
        bid = None  # stop buying
    if inventory <= -q_max:
        ask = None  # stop selling

    return {
        'mode': mode,
        'bid': bid,
        'ask': ask,
        'alpha': alpha,
        'spread': (ask - bid) if bid is not None and ask is not None else None,
        'reservation_price': res_p,
    }

## Simulation Over Sample Day

In [ ]:
# Simulate market making decisions over a sample period
# Use XGBoost probabilities as proxy for model predictions

if 'xgb_prob' not in features.columns:
    features['xgb_prob'] = 0.5  # fallback

vpin = features.get('vpin', pd.Series(0.5, index=features.index))
vpin_mean = vpin.rolling(1440, min_periods=60).mean().fillna(0.5)
vpin_std = vpin.rolling(1440, min_periods=60).std().fillna(0.1)
regime_score = regimes['regime_score'].reindex(features.index, method='ffill').fillna(0)

# Take last 1440 bars (~1 day)
sample = features.iloc[-1440:].copy()
capital = 1000  # USD
inventory = 0

results = []
for i, (ts, row) in enumerate(sample.iterrows()):
    # Simulate 15-min windows cycling
    time_remaining = 1.0 - (i % 15) / 15.0

    action = compute_action(
        p_up=row.get('xgb_prob', 0.5),
        vpin=vpin.get(ts, 0.5),
        vpin_mean=vpin_mean.get(ts, 0.5),
        vpin_std=vpin_std.get(ts, 0.1),
        regime_score=regime_score.get(ts, 0.0),
        time_remaining=time_remaining,
        inventory=inventory,
        capital=capital
    )
    results.append(action)

results_df = pd.DataFrame(results, index=sample.index)
print(f'Simulated {len(results_df)} quoting decisions')

## Metrics

In [ ]:
# Mode distribution
print('Mode distribution:')
print(results_df['mode'].value_counts(normalize=True).round(3))

# Spread distribution
spreads = results_df['spread'].dropna()
if len(spreads) > 0:
    print(f'\nSpread stats:')
    print(f'  Mean:   {spreads.mean():.4f}')
    print(f'  Median: {spreads.median():.4f}')
    print(f'  Min:    {spreads.min():.4f}')
    print(f'  Max:    {spreads.max():.4f}')

# Quotes active rate
active = results_df['bid'].notna() | results_df['ask'].notna()
print(f'\nQuotes active: {active.mean():.1%}')

In [ ]:
# Spread distribution histogram
if len(spreads) > 0:
    fig, axes = plt.subplots(1, 2, figsize=(10, 4))

    axes[0].hist(spreads.clip(0, 0.2), bins=50, edgecolor='black', alpha=0.7)
    axes[0].set_xlabel('Spread')
    axes[0].set_ylabel('Count')
    axes[0].set_title('Spread Distribution')

    mode_counts = results_df['mode'].value_counts()
    axes[1].bar(mode_counts.index, mode_counts.values, edgecolor='black', alpha=0.7)
    axes[1].set_ylabel('Count')
    axes[1].set_title('Mode Distribution')
    plt.setp(axes[1].xaxis.get_majorticklabels(), rotation=30)

    plt.tight_layout()
    plt.show()